# Artmatcher

This notebook demonstrates the artmatcher for determining whether a query image matches a reference artwork. The artmatcher makes use of a **Random Forest classifier**.

The classifier combines three feature categories:
- **Distribution features**: Statistics from embedding similarity scores (gaps, z-scores, decay rates)
- **ORB features**: Geometric verification using ORB keypoints and homography
- **Perceptual hashes**: Multiple hash distances (pHash, dHash, wHash, aHash)

These 23 features (detailed features see below in the notebook) are fed into a **calibrated Random Forest** trained on labeled examples from the web UI.
The random forest outputs a confidence score, which is to say how confident the model is that two images are a 'match', given their features.

Main technologies used:
- [Scikit-learn Random Forest](https://scikit-learn.org/stable/modules/ensemble.html#forests-of-randomized-trees) - Ensemble classifier with probability calibration
- [OpenCV ORB](https://docs.opencv.org/4.x/d1/d89/tutorial_py_orb.html) - Oriented FAST and Rotated BRIEF for geometric verification
- [Perceptual Hashing](https://github.com/JohannesBuchner/imagehash) - Image similarity using perceptual hashes
- [Homography](https://docs.opencv.org/4.x/d9/dab/tutorial_homography.html) - Geometric transformation verification

# Setup

### Dependencies

Import all necessary python libraries:

In [6]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

from artmatcher.ml_classifier import FeatureExtractor, MLClassifier

### Configuration

You can test the artmatcher on a set of example images or modify some variabels to let the artmatcher run on your own set of images.


The configurations exist of:
- IMAGES_DIR: the path to where your images are stored.
- QUERY_IMAGE & QUERY_IMAGE_2: The path to which specific image you want to compare in the artmatcher
- REFERENCE_IMAGE & REFERENCE_IMAGE_2: the path to which image you want to use as a reference.
- TRAINED_MODEL_PATH: path to the model you want to use. Always needs a `.pkl` extension.
- LABELED_DATASET_PATH: path to the labeled dataset. Always needs a `.json` extensions.

Run the code block below to load the paths to the set of example images and model:

In [7]:
# --- Example images ---
EXAMPLES_DIR = Path("./examples")

QUERY_IMAGE = EXAMPLES_DIR / "028pc5th0k_tiff.jpg"  # A glashelder photo
QUERY_IMAGE_2 = EXAMPLES_DIR / "1n7xm18d8t_tiff.jpg"

REFERENCE_IMAGE = (EXAMPLES_DIR / "joachim-patinir_rest-during-the-flight-to-egypt.jpg")  # A wikiart painting
REFERENCE_IMAGE_2 = (EXAMPLES_DIR / "notebooks/artmatcher/examples/ancient-greek-painting_boxer-head-by-myron--400.jpg")

# --- Model paths ---
TRAINED_MODEL_PATH = EXAMPLES_DIR / "trained-matcher-model.pkl"
LABELED_DATASET_PATH = Path("./labeling_dataset.json")

Set the paths below to point to your own images and the chosen trained model, and run the block of code to load the paths to the images and model: **ADD MORE INFO IN THE CODE BLOCK BELOW**

In [ ]:
IMAGES_DIR = Path("./examples")

QUERY_IMAGE = IMAGES_DIR / "028pc5th0k_tiff.jpg"  # A glashelder photo
QUERY_IMAGE_2 = IMAGES_DIR / "1n7xm18d8t_tiff.jpg"

REFERENCE_IMAGE = (
    IMAGES_DIR / "joachim-patinir_rest-during-the-flight-to-egypt.jpg"
)  # A wikiart painting
REFERENCE_IMAGE_2 = (
    IMAGES_DIR
    / "notebooks/artmatcher/examples/ancient-greek-painting_boxer-head-by-myron--400.jpg"
)

# --- Model paths ---
TRAINED_MODEL_PATH = EXAMPLES_DIR / "trained-matcher-model.pkl"
LABELED_DATASET_PATH = Path("./labeling_dataset.json")

NameError: name 'Path' is not defined

---

# Part 1: Random Forest Classifier

The classifier decides whether a top embedding match is a **true match** or a false positive. It uses 23 engineered features across three categories:

```mermaid
flowchart TD
  subgraph Input
    emb[Embedding Similarities] --> dist[Distribution Features]
    imgs[Query + Reference Images] --> orb[ORB Features]
    imgs --> hash[Hash Features]
  end
  subgraph "Random Forest Classifier"
    dist --> feat[23-Feature Vector]
    orb --> feat
    hash --> feat
    feat --> rf[Calibrated Random Forest]
    rf --> decision["is_match + confidence"]
  end
```

**Workflow overview:**
1. Compute CLIP embedding similarities between query and reference database
2. Extract 23 features (distribution stats, ORB geometric verification, perceptual hashes)
3. Classify with a trained Random Forest → calibrated probability
4. Threshold the probability to decide match vs. no-match

## 1.1 The 23 Features

The classifier uses these features:

### Distribution (from similarities only):
- sim_0
- sim_1
- sim_2
- gap_01
- gap_02
- gap_12
- mean_1to9
- std_1to9
- sim_0_zscore
- decay_rate
- sim_min
- sim_range
- poly_diff
- sim_ratio_01

### ORB geometric (requires images):
- orb_num_matches
- orb_num_inliers
- orb_confidence
- orb_warp_stretch
- inlier_match_ratio

### Perceptual hashes (requires images):
- phash_distance
- dhash_distance
- whash_distance
- ahash_distance

## 1.2 Feature Extraction

Extract features from similarity scores and image pairs.

In [8]:
extractor = FeatureExtractor()

# Distribution features only (fast, no images needed)
# (this would normally be retrieved from actual
example_similarities = [0.82, 0.51, 0.43, 0.41, 0.38, 0.35, 0.33, 0.30, 0.28, 0.25]

dist_features = extractor.extract_distribution_features(example_similarities)

print("Distribution features for example similarities:")
for name, value in dist_features.items():
    print(f"  {name}: {value:.4f}")

Distribution features for example similarities:
  sim_0: 0.8200
  sim_1: 0.5100
  sim_2: 0.4300
  gap_01: 0.3100
  gap_02: 0.3900
  gap_12: 0.0800
  mean_1to9: 0.3600
  std_1to9: 0.0770
  sim_0_zscore: 5.9718
  decay_rate: -0.0293
  sim_min: 0.2500
  sim_range: 0.5700
  poly_diff: 0.3368
  sim_ratio_01: 1.6078


In [9]:
# Full feature extraction (distribution + ORB + hashes) requires image paths
if QUERY_IMAGE.exists() and REFERENCE_IMAGE.exists():
    all_features = extractor.extract_all_features(
        example_similarities,
        glashelder_path=str(QUERY_IMAGE),
        wikiart_path=str(REFERENCE_IMAGE),
    )

    print("All 23 features:")
    for name, value in all_features.items():
        print(f"  {name}: {value:.4f}")
else:
    print(
        f"Place example images at:\n  {QUERY_IMAGE}\n  {REFERENCE_IMAGE}\nto extract ORB and hash features."
    )

Place example images at:
  examples/028pc5th0k_tiff.jpg
  examples/joachim-patinir_rest-during-the-flight-to-egypt.jpg
to extract ORB and hash features.


## 1.3 Visualize Example Images

In [10]:
if QUERY_IMAGE.exists() and REFERENCE_IMAGE.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(Image.open(QUERY_IMAGE))
    axes[0].set_title("Query (glashelder)")
    axes[0].axis("off")

    axes[1].imshow(Image.open(REFERENCE_IMAGE))
    axes[1].set_title("Reference (wikiart)")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print(f"Place example images at:\n  {QUERY_IMAGE}\n  {REFERENCE_IMAGE}")

Place example images at:
  examples/028pc5th0k_tiff.jpg
  examples/joachim-patinir_rest-during-the-flight-to-egypt.jpg


---

## 1.4 Training the Classifier

Training requires a labeled dataset JSON (produced by the `artmatcher serve` web UI) with four categories:
- **positives**: classifier correctly predicted match
- **negatives**: classifier correctly predicted no match
- **false_positives**: classifier said match, but was wrong → trained as negative
- **false_negatives**: classifier said no match, but was wrong → trained as positive

Each entry has three fields:
- `glashelder_path`: path to the query image
- `wikiart_path`: path to the reference painting
- `similarities`: top-k embedding similarity scores (list of floats, descending)

```json
{
  "positives": [
    {
      "glashelder_path": "/data/glashelder/img_001.jpg",
      "wikiart_path": "/data/wikiart/monet_water_lilies.jpg",
      "similarities": [0.91, 0.42, 0.38, 0.35, 0.33, 0.31, 0.29, 0.27, 0.25, 0.23]
    }
  ],
  "negatives": [
    {
      "glashelder_path": "/data/glashelder/img_002.jpg",
      "wikiart_path": "/data/wikiart/picasso_guernica.jpg",
      "similarities": [0.45, 0.43, 0.42, 0.41, 0.40, 0.39, 0.38, 0.37, 0.36, 0.35]
    }
  ],
  "false_positives": [
    {
      "glashelder_path": "/data/glashelder/img_003.jpg",
      "wikiart_path": "/data/wikiart/renoir_luncheon.jpg",
      "similarities": [0.72, 0.44, 0.40, 0.38, 0.36, 0.34, 0.32, 0.30, 0.28, 0.26]
    }
  ],
  "false_negatives": [
    {
      "glashelder_path": "/data/glashelder/img_004.jpg",
      "wikiart_path": "/data/wikiart/vermeer_girl_pearl.jpg",
      "similarities": [0.58, 0.41, 0.39, 0.37, 0.35, 0.33, 0.31, 0.29, 0.27, 0.25]
    }
  ]
}
```

In [11]:
if LABELED_DATASET_PATH.exists():
    classifier = MLClassifier(threshold=0.5)

    metrics = classifier.train(
        dataset_path=str(LABELED_DATASET_PATH),
        n_estimators=100,
        cv_folds=5,
        extract_image_features=True,
        workers=4,
    )

    print(
        f"Cross-validation accuracy: {metrics.cv_accuracy:.3f} ± {metrics.cv_std:.3f}"
    )
    print(
        f"Samples: {metrics.n_samples} ({metrics.n_positives} pos, {metrics.n_negatives} neg)"
    )
    print("\nTop features by importance:")
    for name, imp in sorted(metrics.feature_importance.items(), key=lambda x: -x[1])[
        :10
    ]:
        print(f"  {name}: {imp:.4f}")

    classifier.save(str(TRAINED_MODEL_PATH))
    print(f"\nModel saved to {TRAINED_MODEL_PATH}")
else:
    print(f"No labeled dataset found at {LABELED_DATASET_PATH}")
    print("Use `uv run glashelder artmatcher serve` to label examples first.")

No labeled dataset found at labeling_dataset.json
Use `uv run glashelder artmatcher serve` to label examples first.


## 1.5 Classification

Load a trained model and classify an image pair.

In [ ]:
if TRAINED_MODEL_PATH.exists():
    classifier = MLClassifier.load(str(TRAINED_MODEL_PATH), threshold=0.81)
    print(f"Loaded model (threshold={classifier.threshold:.2f})")

    result = classifier.classify(
        similarities=example_similarities,
        glashelder_path=str(QUERY_IMAGE) if QUERY_IMAGE.exists() else None,
        wikiart_path=str(REFERENCE_IMAGE) if REFERENCE_IMAGE.exists() else None,
    )

    print(f"\nIs match: {result.is_match}")
    print(f"Confidence: {result.confidence:.4f}")
    print(f"Threshold: {result.threshold:.4f}")
else:
    print(f"No trained model at {TRAINED_MODEL_PATH}")
    print(
        "Train one in section 1.4 above, or via CLI: uv run glashelder artmatcher train-classifier"
    )

## 1.6 Visualize Feature Importance

If a model is loaded, visualize which features matter most.

In [ ]:
if TRAINED_MODEL_PATH.exists():
    clf = MLClassifier.load(str(TRAINED_MODEL_PATH))

    # Get feature importance from the base RF inside CalibratedClassifierCV
    base_rf = clf._model.calibrated_classifiers_[0].estimator
    importances = base_rf.feature_importances_
    feature_importance = dict(zip(MLClassifier.FEATURE_ORDER, importances, strict=True))

    sorted_features = sorted(feature_importance.items(), key=lambda x: x[1])
    names, importances = zip(*sorted_features, strict=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(names, importances)
    ax.set_xlabel("Importance")
    ax.set_title("Random Forest Feature Importance")
    plt.tight_layout()
    plt.show()
else:
    print(f"No trained model at {TRAINED_MODEL_PATH}")

---

## CLI Commands

The full pipeline is also available via the CLI:

```bash
# Label examples interactively (builds labeling_dataset.json)
uv run glashelder artmatcher serve

# Train a classifier from labeled data
uv run glashelder artmatcher train-classifier

# Batch classify all images
uv run glashelder artmatcher batch-classify
```